# Event-Driven Market Reaction & Volatility Analysis System
## Machine Learning Pipeline (PV Project Colab)

**Author:** User / Student \n
**Context:** Czech high school PV project

### Project Overview
This educational notebook documents the creation and training of the classification model used to predict abnormal market volatility based on financial market data and real-world sentiment analysis.

#### Dataset Sources
1. **Market Data**: 5-Year historical price and volume data fetched from Yahoo Finance API for 25 target tickers.
2. **News Data**: Real-world textual data fetched from Reddit's public APIs (r/investing, r/StockMarket), alongside RSS feeds.

#### Features Engineered
- `daily_volatility`: Approximation of True Range using High/Low/Open.
- `macro_sentiment`: NLTK VADER sentiment analysis applied to general financial news headlines.
- `macro_engagement`: Popularity score derived from Reddit upvotes and comment counts.

#### Target Variable
- `abnormal_movement`: 1 if the stock moved more than 1.5% in the next 24 hours (class `1`), else 0 (class `0`).


In [ ]:
# 1. Setup Environment
!pip install scikit-learn pandas numpy matplotlib seaborn nltk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

### Note on running this locally/Colab
Since we run this in Colab, please upload the `ml_dataset.csv` generated by `feature_engineering.py` manually to the Colab files section.

In [ ]:
# Load preprocessed data
try:
    df = pd.read_csv('ml_dataset.csv')
    print(f"Data loaded successfully! Shape: {df.shape}")
except FileNotFoundError:
    print("Please upload ml_dataset.csv to use this notebook.")
    print("You can generate it by running `python feature_engineering.py` from the main project.")


In [ ]:
if 'df' in locals():
    feature_cols = [
        'open', 'volume', 'daily_volatility',
        'macro_sentiment', 'macro_engagement', 'macro_news_count',
        'specific_sentiment', 'specific_engagement', 'specific_news_count'
    ]
    
    X = df[feature_cols].fillna(0)
    y = df['abnormal_movement']
    
    # 80/20 Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Model Choice: RandomForestClassifier
    # Chosen because it naturally handles non-linear variables and limits overfitting (via ensemble bagging)
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    
    print("Training model...")
    model.fit(X_train, y_train)
    print("Done!")


### Evaluation & Visualizations
Check how accurately the model maps sentiment to actual price reactions.

In [ ]:
if 'model' in locals():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    
    # Generate Confusion Matrix visualization
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix: Predicting Market Spikes')
    plt.xlabel('Predicted (0=Stable, 1=Spike)')
    plt.ylabel('Actual (0=Stable, 1=Spike)')
    plt.show()


In [ ]:
if 'model' in locals():
    print("--- Overfitting Check via Cross-Validation ---")
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"5-Fold CV Scores: {cv_scores}")
    print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
    print(f"Test set accuracy: {model.score(X_test, y_test):.4f}")
    
    if (model.score(X_train, y_train) - model.score(X_test, y_test)) < 0.15:
        print("\n✅ Conclusion: Model generalizing well. No severe overfitting detected.")


In [ ]:
if 'model' in locals():
    # Feature Importances
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 5))
    importances.plot(kind='bar', color='darkgreen')
    plt.title('Feature Importances for Volatility Prediction')
    plt.ylabel('Relative Importance')
    plt.tight_layout()
    plt.show()
